# 06 · Swap one slot at a time

A foundation-model design is five choices (data, event representation,
sequence architecture, objective, readout and adaptation). To learn what one
choice buys, change only that one and score every variant the same way. This
notebook runs the [building-block ablation](../../benchmarks/ablations/building_blocks)
recipe on CPU and reads its table.

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

from tfm.config import load_recipe_config
from tfm.runtime.common import repo_root
from tfm.runtime.compare import load_variants

RECIPE = repo_root() / "benchmarks" / "ablations" / "building_blocks"
RUN_DIR = repo_root() / "outputs" / "notebooks" / "building_blocks"
base = load_recipe_config(RECIPE / "config.yaml")
variants = load_variants(RECIPE / "variants.yaml")
for v in variants:
    print(f"{v.name:<20} {v.slot:<22} {dict(v.overrides) or '(base)'}")

## Fair by construction

Every variant keeps the base `data` section and seed. The book, the cutoffs,
the label windows, and the probe's user split are therefore identical, and
only the swapped slot differs. `apply_overrides` validates each variant as a
full config, so a pairing the kit refuses fails here, before any training.

In [ ]:
from tfm.runtime.compare import apply_overrides

for v in variants:
    config = apply_overrides(base, v.overrides)
    c = config.model.composition
    print(f"{v.name:<20} {config.features.tokenizer:<16} {c.granularity.type:<22} "
          f"{c.position:<10} {c.objective}+{config.objective.auxiliary} {c.readout}")

## Run three variants

The whole table takes about a minute on CPU; three rows keep this notebook
quick. Use `compare.py --seeds 42 43 44` for the full table over seeds.

In [ ]:
from IPython.display import Markdown

from tfm.runtime.compare import comparison_markdown, run_comparison

chosen = [v for v in variants if v.name in {"v2_hybrid", "flat_decoder", "mean_readout"}]
results = run_comparison(base, chosen, RUN_DIR, seeds=[base.seed])
Markdown(comparison_markdown(results))

## Reading the table

- `raw` is the same model-free baseline in every row; if it moves, the data
  changed, not the model.
- `embedding` is what the swapped slot changed.
- `combined` asks whether the embedding adds anything to the raw features.
- The last column is the number of evaluation positives. At four, no
  difference in this table means anything: it shows the method, not a ranking.

To compare your own choice, add a row to `variants.yaml` and rerun.